# 01: Quality Control and Exploratory Data Analysis

This notebook performs quality control checks (gaps, duplicates) and exploratory data analysis.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from loguru import logger

from src import io, qc

# Set random seed and plotting style
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style('whitegrid')

logger.info("QC and EDA notebook initialized")

## 1. Load Data

In [ ]:
# Load data from previous step
data_path = Path('../data/processed/intake_data.parquet')
df = io.load_raw_data(data_path)

logger.info(f"Loaded data with shape: {df.shape}")
df.head()

## 2. Quality Control Checks

In [ ]:
# Detect missing values
missing_summary = qc.detect_missing_values(df)
print("\nMissing Values:")
print(missing_summary)

In [ ]:
# Detect duplicates
duplicates_df, n_duplicates = qc.detect_duplicates(df)
print(f"\nNumber of duplicates: {n_duplicates}")
if n_duplicates > 0:
    print("\nDuplicate rows:")
    print(duplicates_df.head())

In [ ]:
# Detect time gaps
if 'timestamp' in df.columns:
    gaps = qc.detect_time_gaps(df, 'timestamp', expected_freq='1H')
    print(f"\nNumber of time gaps: {len(gaps)}")
    if len(gaps) > 0:
        print("\nTime gaps detected:")
        print(gaps)

In [ ]:
# Generate comprehensive QC report
qc_results = qc.qc_report(df, time_column='timestamp')
print("\nQC Report:")
for key, value in qc_results.items():
    if key not in ['missing_values', 'time_gaps', 'dtypes']:
        print(f"  {key}: {value}")

## 3. Clean Data

In [ ]:
# Remove duplicates if any
df_clean = qc.remove_duplicates(df)
logger.info(f"Clean data shape: {df_clean.shape}")

## 4. Exploratory Data Analysis

In [ ]:
# Time series plot
if 'timestamp' in df_clean.columns and 'value' in df_clean.columns:
    plt.figure(figsize=(14, 6))
    plt.plot(df_clean['timestamp'], df_clean['value'])
    plt.xlabel('Time')
    plt.ylabel('Value')
    plt.title('Time Series Plot')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('../outputs/timeseries_plot.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Distribution plot
if 'value' in df_clean.columns:
    plt.figure(figsize=(10, 6))
    plt.hist(df_clean['value'], bins=50, edgecolor='black')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.title('Value Distribution')
    plt.tight_layout()
    plt.savefig('../outputs/distribution_plot.png', dpi=300, bbox_inches='tight')
    plt.show()

## 5. Save Cleaned Data

In [ ]:
# Save cleaned data
output_path = Path('../data/processed/qc_data.parquet')
io.save_processed_data(df_clean, output_path, format='parquet')

logger.success(f"Cleaned data saved to {output_path}")

## 6. Log Parameters

In [ ]:
params = {
    'random_state': RANDOM_STATE,
    'initial_samples': qc_results['n_rows'],
    'final_samples': len(df_clean),
    'duplicates_removed': qc_results['n_duplicates'],
}

logger.info(f"QC parameters: {params}")
print("\nQC Parameters:")
for key, value in params.items():
    print(f"  {key}: {value}")